In [3]:
import os
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
from argparse import Namespace
import torch
import requests
import numpy as np
import cv2
from PIL import Image
import utils_model
import utils_gradio
import utils_attn
device_map = "auto"

In [ ]:
model_name_or_path = "llava-hf/llava-1.5-7b-hf"

args = Namespace(
    model_name_or_path=model_name_or_path,
    load_4bit=False,
    load_8bit=False,
    device_map=device_map,
)
processor, model = utils_model.get_processor_model(args)

In [ ]:
processor

In [ ]:
model

In [4]:
# model_name_or_path = "Qwen/Qwen2.5-VL-3B-Instruct"

# args = Namespace(
#     model_name_or_path=model_name_or_path,
#     load_4bit=False,
#     load_8bit=False,
#     device_map=device_map,
# )
# processor2, model2 = utils_model.get_processor_model(args)

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_name_or_path = "Qwen/Qwen2-VL-7B-Instruct"
model = AutoModelForImageTextToText.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map=device_map
)

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.07steps/s]


In [6]:
model

Qwen2VLForConditionalGeneration(
  (visual): Qwen2VisionTransformerPretrainedModel(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2VLVisionBlock(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): VisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): VisionMlp(
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (act): QuickGELUActivation()
          (fc2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
    )
    (merger): PatchMerger(
      (ln_q): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (mlp): Seq

In [7]:
model_name_or_path = "Qwen/Qwen2-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(model_name_or_path)
processor

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Qwen2VLProcessor:
- image_processor: Qwen2VLImageProcessor {
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "max_pixels": 12845056,
  "merge_size": 2,
  "min_pixels": 3136,
  "patch_size": 14,
  "processor_class": "Qwen2VLProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 12845056,
    "shortest_edge": 3136
  },
  "temporal_patch_size": 2
}

- tokenizer: Qwen2TokenizerFast(name_or_path='Qwen/Qwen2-VL-7B-Instruct', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start

In [ ]:
utils_gradio.processor = processor
utils_gradio.model = model